# 🛣️ RoadGuardAI — Genuine Pothole Detection Model Training
## Stage 3: YOLOv8n Fine-Tuning, Test Set Evaluation & Checkpoint Export

This Google Colab notebook provides an end-to-end, reproducible training workflow for **RoadGuardAI**.

### 📋 Student / Researcher Workflow (10 Steps):
1. **Runtime Selection:** Ensure GPU is selected (`Runtime` → `Change runtime type` → `T4 GPU`).
2. **Environment Verification:** Check GPU availability, PyTorch, and CUDA versions.
3. **Repository Setup:** Clone the RoadGuardAI repository and verify scripts.
4. **Dataset Acquisition:** Acquire the Roboflow Pothole Detection dataset.
5. **Dataset Preparation:** Run group-level isolation splitting (70% train, 20% val, 10% test).
6. **Dataset Audit:** Verify image integrity, bounding box normalization, and zero cross-split leakage.
7. **Model Training:** Fine-tune YOLOv8n with `ml/configs/roadguard.yaml` (320x320, seed 42, 50 epochs).
8. **Held-Out Test Evaluation:** Compute Precision, Recall, mAP@0.5, and mAP@0.5:0.95.
9. **Artifact Packaging:** Save `best.pt`, training metadata, confusion matrix, and curves.
10. **Model Download:** Download the trained checkpoint for Stage 4 ONNX export and mobile deployment.

### Step 1 & 2: Environment Verification & GPU Check

In [ ]:
# Step 2: Check GPU and Python Environment
import sys
import os
import torch

print(f"Python Version:  {sys.version.split()[0]}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version:    {torch.version.cuda}")
    !nvidia-smi
else:
    print("\n⚠️ [WARNING] No GPU detected! Please go to Runtime -> Change runtime type -> Select T4 GPU.")

### Step 2b: Install Dependencies

In [ ]:
# Install required packages
!pip install --upgrade pip
!pip install "ultralytics>=8.4.0" "roboflow>=1.1.0" "onnx>=1.12.0,<2.0.0" "onnxruntime>=1.16.0" onnxslim pyyaml pillow

### Step 3: Clone or Setup Repository

In [ ]:
# Step 3: Clone RoadGuardAI repository (or navigate to existing directory)
REPO_URL = "https://github.com/ANURODHOP/RoadGuardAI.git"
REPO_DIR = "RoadGuardAI"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL

%cd $REPO_DIR
print("Current Directory:", os.getcwd())

# Verify that pipeline scripts are present
required_scripts = [
    "ml/scripts/prepare_dataset.py",
    "ml/scripts/audit_dataset.py",
    "ml/scripts/dataset_statistics.py",
    "ml/scripts/train.py",
    "ml/scripts/evaluate.py",
    "ml/configs/roadguard.yaml"
]
missing = [s for s in required_scripts if not os.path.exists(s)]
if missing:
    print(f"\n⚠️ [WARNING] Missing from cloned repo: {missing}")
    print("Make sure latest commits are pushed to the GitHub repository branch.")
else:
    print("\n[OK] All ML pipeline scripts and configs verified:")
    !ls -la ml/scripts/
    !ls -la ml/configs/

### Step 4: Dataset Acquisition

We acquire the **Roboflow Pothole Detection** dataset (`roboflow-100/pothole-detection-gwyag`, v2).
- License: **CC BY 4.0** (Open commercial & academic use with attribution).
- Note: Enter your free Roboflow API key when prompted (never hardcode credentials into notebooks).

In [ ]:
# Step 4: Download dataset from Roboflow Universe
import getpass
from roboflow import Roboflow

ROBOFLOW_WORKSPACE = "roboflow-100"
ROBOFLOW_PROJECT   = "pothole-detection-gwyag"
ROBOFLOW_VERSION   = 2
RAW_DATASET_DIR    = "ml/datasets/raw_pothole"

# Enter your Roboflow API key (free at https://app.roboflow.com/)
api_key = os.environ.get("ROBOFLOW_API_KEY")
if not api_key:
    api_key = getpass.getpass("Enter your Roboflow API Key (input hidden): ")

rf = Roboflow(api_key=api_key)
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
dataset = project.version(ROBOFLOW_VERSION).download("yolov8", location=RAW_DATASET_DIR)

print("\n[OK] Raw dataset acquired at:", dataset.location)
!ls -la $RAW_DATASET_DIR

### Step 5: Prepare Dataset with Group-Level Isolation

We run `ml/scripts/prepare_dataset.py` to:
1. Ingest Roboflow images and labels across all split folders.
2. Cluster sequential video frames using `extract_group_key()` to prevent temporal data leakage.
3. Partition into 70% train, 20% validation, 10% test using deterministic `seed=42`.
4. Flag exact duplicates via MD5 and near-duplicates via 8x8 difference hash (dHash).
5. Output canonical `ml/datasets/roadguard/dataset.yaml` with single class `0: pothole`.

In [ ]:
# Step 5: Ingest raw dataset and partition with group isolation
!python ml/scripts/prepare_dataset.py \
    --source-dir ml/datasets/raw_pothole \
    --output-dir ml/datasets/roadguard \
    --seed 42 \
    --dhash-threshold 10

### Step 6: Dataset Integrity Audit & Statistics

Before training, we run `ml/scripts/audit_dataset.py` and `ml/scripts/dataset_statistics.py`.
This verifies:
- 0 corrupted images
- Complete image-label parity (0 orphan or missing labels)
- Bounding boxes normalized in `[0.0, 1.0]` with class ID 0
- 0 cross-split image leakage
- Quantitative box scale & aspect ratio distributions

In [ ]:
# Step 6a: Run dataset integrity audit
!python ml/scripts/audit_dataset.py --dataset-dir ml/datasets/roadguard

# Step 6b: Generate quantitative dataset statistics
!python ml/scripts/dataset_statistics.py --dataset-dir ml/datasets/roadguard

### Step 7: Train YOLOv8n Pothole Detection Model

We train using `ml/scripts/train.py` which loads parameters from `ml/configs/roadguard.yaml`:
- **Architecture:** `YOLOv8n` (Nano, ~3.2M parameters)
- **Resolution:** `320x320` (exact match with mobile input tensor contract)
- **Epochs:** `50` with early stopping patience of 15
- **Batch Size:** `16` (on Colab GPU) or `8` (on CPU)
- **Optimizer:** `AdamW` (`lr0=0.001`, `lrf=0.01` with cosine schedule)
- **Seed:** `42` (fully reproducible)
- **Augmentations:** Road-specific augmentations (HSV lighting, horizontal flip, mosaic=0.5; vertical flip & mixup disabled)

In [ ]:
# Step 7: Train YOLOv8n pothole detector
device_arg = "0" if torch.cuda.is_available() else "cpu"
batch_arg  = 16 if torch.cuda.is_available() else 8

!python ml/scripts/train.py \
    --config ml/configs/roadguard.yaml \
    --data ml/datasets/roadguard/dataset.yaml \
    --epochs 50 \
    --batch $batch_arg \
    --device $device_arg \
    --imgsz 320 \
    --seed 42

### Step 8: Comprehensive Evaluation on Held-Out Test Split

We evaluate the best trained checkpoint (`ml/models/best.pt`) on the **held-out test split**.
The test split was never seen during training or validation.

In [ ]:
# Step 8: Run independent evaluation on held-out test split
!python ml/scripts/evaluate.py \
    --weights ml/models/best.pt \
    --data ml/datasets/roadguard/dataset.yaml \
    --split test \
    --imgsz 320 \
    --device $device_arg \
    --output-dir ml/runs/eval_test

### Step 8b: Visualize Training Curves & Confusion Matrix

In [ ]:
from IPython.display import Image, display
from pathlib import Path

run_dir = Path("ml/runs/pothole_yolov8n")

# Display training curves and confusion matrix if available
for plot_name in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    plot_file = run_dir / plot_name
    if plot_file.exists():
        print(f"\n=== {plot_name} ===")
        display(Image(filename=str(plot_file), width=700))

# Display sample test batch predictions
eval_dir = Path("ml/runs/eval_test")
for pred_plot in eval_dir.glob("val_batch*_pred.jpg"):
    print(f"\n=== Prediction Sample: {pred_plot.name} ===")
    display(Image(filename=str(pred_plot), width=700))
    break

### Step 9: Package Training Artifacts

In [ ]:
# Step 9: Create a zip archive of model weights and metadata for download
import zipfile

archive_name = "roadguard_stage3_artifacts.zip"
with zipfile.ZipFile(archive_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    # Model checkpoint
    if os.path.exists("ml/models/best.pt"):
        zipf.write("ml/models/best.pt", arcname="best.pt")
    # Training metadata
    if os.path.exists("ml/models/training_metadata.json"):
        zipf.write("ml/models/training_metadata.json", arcname="training_metadata.json")
    # Evaluation report
    if os.path.exists("ml/runs/eval_test/evaluation_report.json"):
        zipf.write("ml/runs/eval_test/evaluation_report.json", arcname="evaluation_report.json")
    # Key visualization plots
    for plot_path in run_dir.glob("*.png"):
        zipf.write(str(plot_path), arcname=f"plots/{plot_path.name}")

print(f"[OK] Artifacts archived to: {archive_name} ({os.path.getsize(archive_name) / 1024 / 1024:.2f} MB)")
!ls -lh $archive_name

### Step 10: Download Model Checkpoint to Local Machine

In [ ]:
# Step 10: Download the artifact archive to your computer
try:
    from google.colab import files
    files.download("roadguard_stage3_artifacts.zip")
    print("Download initiated! Check your browser's download tray.")
except ImportError:
    print("Running outside Google Colab. The archive is located at: roadguard_stage3_artifacts.zip")